# Construction Safety Monitor

**Model:** YOLOv8s  
**Task:** PPE compliance detection  
**Dataset:** Roboflow Construction Safety Dataset (downloaded manually) + our own site photos

### Classes
| ID | Class | Safe? |
|----|-------|-------|
| 0 | `hardhat` | Safe |
| 1 | `no_hardhat` | VIOLATION |
| 2 | `vest` | Safe |
| 3 | `no_vest` | VIOLATION |
| 4 | `person` | Neutral |


In [1]:
import os
import zipfile
from google.colab import files

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
DATASET_DIR = '/content/dataset'

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

print(f'Extracted to: {DATASET_DIR}')

# find the data.yaml (it might be nested inside a subfolder)
DATA_YAML = None
for root, dirs, fnames in os.walk(DATASET_DIR):
    for fname in fnames:
        if fname == 'data.yaml':
            DATA_YAML = os.path.join(root, fname)
            DATASET_DIR = root  # reset to wherever data.yaml lives
            break

if DATA_YAML is None:
    raise FileNotFoundError('data.yaml not found. Make sure you downloaded YOLOv8 format from Roboflow.')

print(f'data.yaml found at: {DATA_YAML}')

Saving Construction Site Safety.v30-raw-images_latestversion.yolov8.zip to Construction Site Safety.v30-raw-images_latestversion.yolov8.zip
Extracted to: /content/dataset
data.yaml found at: /content/dataset/data.yaml


In [2]:
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print(cfg['names'])
print('Total classes:', cfg['nc'])

['Excavator', 'Gloves', 'Hardhat', 'Ladder', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'SUV', 'Safety Cone', 'Safety Vest', 'bus', 'dump truck', 'fire hydrant', 'machinery', 'mini-van', 'sedan', 'semi', 'trailer', 'truck and trailer', 'truck', 'van', 'vehicle', 'wheel loader']
Total classes: 25


In [3]:
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print('Classes:', cfg['names'])
print('Num classes:', cfg['nc'])

# fix paths in data.yaml to point to absolute paths so YOLO can find them
cfg['train'] = os.path.join(DATASET_DIR, 'train', 'images')
cfg['val']   = os.path.join(DATASET_DIR, 'valid', 'images')
cfg['test']  = os.path.join(DATASET_DIR, 'test',  'images')

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f)

print('data.yaml updated with absolute paths.')

Classes: ['Excavator', 'Gloves', 'Hardhat', 'Ladder', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'Person', 'SUV', 'Safety Cone', 'Safety Vest', 'bus', 'dump truck', 'fire hydrant', 'machinery', 'mini-van', 'sedan', 'semi', 'trailer', 'truck and trailer', 'truck', 'van', 'vehicle', 'wheel loader']
Num classes: 25
data.yaml updated with absolute paths.


## 2. Add Your Own Site Photos

**The label file format** (one line per object in the image):
```
class_id  center_x  center_y  width  height
```

**If you don't have your photos ready yet**, you can skip this cell and come back to it. The rest of the notebook will still run on the Roboflow data alone.

In [ ]:
import shutil
import glob

TRAIN_IMG_DIR = os.path.join(DATASET_DIR, 'train', 'images')
TRAIN_LBL_DIR = os.path.join(DATASET_DIR, 'train', 'labels')

print('Select a zip containing your annotated photos and their .txt label files...')
print('(Each image must have a matching .txt file with the same name)')
print()

custom_upload = files.upload()
custom_zip = list(custom_upload.keys())[0]

CUSTOM_TMP = '/content/custom_tmp'
os.makedirs(CUSTOM_TMP, exist_ok=True)

with zipfile.ZipFile(custom_zip, 'r') as z:
    z.extractall(CUSTOM_TMP)

# collect all images and labels from the extracted folder
custom_images = glob.glob(os.path.join(CUSTOM_TMP, '**', '*.jpg'), recursive=True)
custom_images += glob.glob(os.path.join(CUSTOM_TMP, '**', '*.jpeg'), recursive=True)
custom_images += glob.glob(os.path.join(CUSTOM_TMP, '**', '*.png'), recursive=True)

copied_images = 0
copied_labels = 0
skipped = 0

for img_path in custom_images:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(os.path.dirname(img_path), stem + '.txt')

    if not os.path.exists(lbl_path):
        # also check if label is in a sibling 'labels' folder
        alt_lbl = img_path.replace('images', 'labels').replace(
            os.path.splitext(img_path)[1], '.txt'
        )
        if os.path.exists(alt_lbl):
            lbl_path = alt_lbl
        else:
            print(f'  skipping {os.path.basename(img_path)} - no matching label file found')
            skipped += 1
            continue

    dest_img = os.path.join(TRAIN_IMG_DIR, 'custom_' + os.path.basename(img_path))
    dest_lbl = os.path.join(TRAIN_LBL_DIR, 'custom_' + stem + '.txt')

    shutil.copy(img_path, dest_img)
    shutil.copy(lbl_path, dest_lbl)
    copied_images += 1
    copied_labels += 1

print(f'Added {copied_images} custom images to training set')
if skipped > 0:
    print(f'Skipped {skipped} images (no label file found)')
print(f'Total training images now: {len(glob.glob(os.path.join(TRAIN_IMG_DIR, "*")))}')

Select a zip containing your annotated photos and their .txt label files...
(Each image must have a matching .txt file with the same name)

